In [1]:
%load_ext autoreload
%autoreload 2
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
cell_protein_df = pd.read_csv('../data/MIBI/raw/cell_protein_data.csv')
patient_df = pd.read_csv('../data/MIBI/raw/patient_info.csv')

In [3]:
import sys
import os
import numpy as np
from torch_geometric.loader import DataLoader

sys.path.append(os.path.abspath('../src'))
from utils.data_utils import split_dataset, split_indices
from dataset.placenta import PlacentaDatasetHypergraph
from dataset.extend import ExtendedDataset
from dataset.mibi import MIBIDataset, MIBISubsetHypergraph

def prepare_dataloaders(args):
    if args.dataset == 'mibi':
        dataset = MIBIDataset(data_folder=args.data_folder, k_hop=args.k_hop,hyperedge_features_list=args.hyperedge_features_list)

        ratios = [float(c) for c in args.train_val_test_ratio.split(':')]
        ratios = tuple([c / sum(ratios) for c in ratios])
        indices = list(range(len(dataset)))

        # NOTE: This is a hack to make sure all sets have all classes.
        all_class_subset1 = [0, 1, 4, 6]
        all_class_subset2 = [7, 8, 10, 11]
        all_class_subset3 = [12, 13, 14, 15]
        indices = list(set(indices) - set(all_class_subset1 + all_class_subset2 + all_class_subset3))
        adjusted_ratios = len(dataset) * np.array(ratios) - np.array([4, 4, 4])
        adjusted_ratios = tuple([c / sum(adjusted_ratios) for c in adjusted_ratios])
        train_indices, val_indices, test_indices = \
            split_indices(indices=indices, splits=adjusted_ratios, random_seed=0)
        train_indices += all_class_subset1
        val_indices += all_class_subset2
        test_indices += all_class_subset3

        print(f'PREPARE_DATALOADERS: {dataset.hyperedge_features_list}')
        train_set = MIBISubsetHypergraph(
            dataset=dataset,
            subset_indices=train_indices)
        val_set = MIBISubsetHypergraph(
            dataset=dataset,
            subset_indices=val_indices)
        test_set = MIBISubsetHypergraph(
            dataset=dataset,
            subset_indices=test_indices)

    min_batch_per_epoch = 5
    desired_len = args.desired_batch_size * min_batch_per_epoch
    if len(train_set) < desired_len:
        train_set = ExtendedDataset(dataset=train_set, desired_len=desired_len)

    train_loader = DataLoader(train_set, batch_size=args.batch_size, num_workers=args.num_workers, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=args.batch_size, num_workers=args.num_workers, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=args.batch_size, num_workers=args.num_workers, shuffle=False)

    return train_loader, val_loader, test_loader, dataset.num_classes

In [ ]:
from argparse import Namespace

args = Namespace(
    dataset='mibi',
    data_folder='../data/MIBI/patchified_all_genes',
    train_val_test_ratio='6:2:2',
    desired_batch_size=16,
    batch_size=1,
    k_hop=1,
    num_workers=4,
    hyperedge_features_list=['gene_expression','diffused_gene_correlation']
)

# Access with dot notation
print(args.dataset)  # 'MIBI'

mibi


In [5]:
# Load the data.
train_loader, val_loader, test_loader, num_classes = prepare_dataloaders(args)

PREPARE_DATALOADERS: ['gene_expression']


In [6]:
dataset = MIBIDataset(data_folder=args.data_folder, k_hop=args.k_hop,hyperedge_features_list=args.hyperedge_features_list)

ratios = [float(c) for c in args.train_val_test_ratio.split(':')]
ratios = tuple([c / sum(ratios) for c in ratios])
indices = list(range(len(dataset)))

# NOTE: This is a hack to make sure all sets have all classes.
all_class_subset1 = [0, 1, 4, 6]
all_class_subset2 = [7, 8, 10, 11]
all_class_subset3 = [12, 13, 14, 15]
indices = list(set(indices) - set(all_class_subset1 + all_class_subset2 + all_class_subset3))
adjusted_ratios = len(dataset) * np.array(ratios) - np.array([4, 4, 4])
adjusted_ratios = tuple([c / sum(adjusted_ratios) for c in adjusted_ratios])
train_indices, val_indices, test_indices = \
    split_indices(indices=indices, splits=adjusted_ratios, random_seed=0)
train_indices += all_class_subset1
val_indices += all_class_subset2
test_indices += all_class_subset3

In [7]:
dataset.hyperedge_features_list

['gene_expression']

In [8]:
train_set = MIBISubsetHypergraph(
    dataset=dataset,
    subset_indices=train_indices)

In [9]:
val_set = MIBISubsetHypergraph(
    dataset=dataset,
    subset_indices=val_indices)
test_set = MIBISubsetHypergraph(
    dataset=dataset,
    subset_indices=test_indices)

In [10]:
train_set.hyperedge_features_list

['gene_expression']

In [11]:
train_set[2]

Extracting hyperedge features: ['gene_expression']
tensor([[ 1.1909,  0.9806,  1.2376,  ...,  8.3184,  1.6400, 13.4306],
        [ 0.0000,  0.0000,  2.2835,  ...,  9.6629,  0.0000, 13.6035],
        [ 1.9848,  1.6344,  2.0626,  ..., 11.6551,  1.2611, 13.2654],
        ...,
        [ 1.8638,  1.9106,  3.5248,  ...,  9.8451,  0.0000, 13.3240],
        [ 0.0000,  0.0000,  0.0000,  ...,  9.8069,  0.0000, 13.6210],
        [ 0.0000,  0.6424,  1.2016,  ...,  6.6724,  0.0000, 13.6700]])


HyperGraphData(x=[88, 26], edge_index=[2, 588], edge_attr=[88, 26], y=1)

In [18]:
train_set[2].x.shape

Extracting hyperedge features: ['gene_expression']
tensor([[ 1.1909,  0.9806,  1.2376,  ...,  8.3184,  1.6400, 13.4306],
        [ 0.0000,  0.0000,  2.2835,  ...,  9.6629,  0.0000, 13.6035],
        [ 1.9848,  1.6344,  2.0626,  ..., 11.6551,  1.2611, 13.2654],
        ...,
        [ 1.8638,  1.9106,  3.5248,  ...,  9.8451,  0.0000, 13.3240],
        [ 0.0000,  0.0000,  0.0000,  ...,  9.8069,  0.0000, 13.6210],
        [ 0.0000,  0.6424,  1.2016,  ...,  6.6724,  0.0000, 13.6700]])


torch.Size([88, 26])

In [13]:
train_set[0]

Extracting hyperedge features: ['gene_expression']
tensor([[ 0.0000,  0.0000,  0.0000,  ...,  4.6859,  0.0000, 10.8328],
        [ 0.0000,  0.0000,  0.0000,  ...,  6.9638,  0.0000, 13.6098],
        [ 0.0000,  0.0000,  0.0000,  ...,  5.9537,  0.0000, 13.5879],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  7.1120,  0.0000, 13.7089],
        [ 0.0000,  0.0000,  0.0000,  ...,  4.7342,  0.0000, 13.7238],
        [ 4.3704,  3.9403,  4.5393,  ...,  6.9709,  3.5008, 12.8335]])


HyperGraphData(x=[79, 26], edge_index=[2, 521], edge_attr=[79, 26], y=1)